<a href="https://colab.research.google.com/github/Bilal574645/LLM_Implementation/blob/main/Day_1_QLora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Take a large AI model(from huggingFace) → make it cheaper to work with(Quantization) → learn how LoRA can modify it without changing the whole model.

In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [2]:
!pip install -q --upgrade bitsandbytes

In [3]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets

In [4]:
import os
import re
import math

from tqdm import tqdm
from datetime import datetime

import torch
import transformers

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed
)

from peft import LoraConfig, PeftModel

In [5]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"

PROJECT_NAME = "price"

RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"

LITE_MODE = False

DATA_USER = "MuhammadBilal"

DATASET_NAME = (
    f"{DATA_USER}/items_prompts_lite"
    if LITE_MODE
    else f"{DATA_USER}/items_prompts_full"
)

FINETUNED_MODEL = "MuhammadBilal/price-2026-09-23_15.10.55-lite"

In [6]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("LLMMYTOKEN")
login(hf_token , add_to_git_credential=True)


In [7]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"

In [8]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto"
)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [9]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto"
)

print(
    f"Memory footprint: "
    f"{base_model.get_memory_footprint() / 1e9:,.1f} GB"
)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Memory footprint: 6.4 GB


In [10]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((3072,), eps=1e-05)
    (

In [11]:
quant_config= BitsAndBytesConfig(
    load_in_8bit=True
)

In [12]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [13]:
print(
    f"Memory footprint: "
    f"{base_model.get_memory_footprint() / 1e9:,.1f} GB"
)

Memory footprint: 3.6 GB


In [14]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [15]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [16]:
print(
    f"Memory footprint: "
    f"{base_model.get_memory_footprint() / 1e9:,.1f} GB"
)

Memory footprint: 2.2 GB


In [17]:
r = 32

lora_q_proj = 3072 * r + 3072 * r
lora_k_proj = 3072 * r + 1024 * r
lora_v_proj = 3072 * r + 1024 * r
lora_o_proj = 3072 * r + 3072 * r

lora_layer = (
    lora_q_proj
    + lora_k_proj
    + lora_v_proj
    + lora_o_proj
)

params = lora_layer * 28

size = (params * 4) / 1_000_000

print(
    f"Total number of params: {params:,} "
    f"and size {size:,.1f}MB"
)

Total number of params: 18,350,080 and size 73.4MB


In [18]:
r = 256

lora_q_proj = 3072 * r + 3072 * r
lora_k_proj = 3072 * r + 1024 * r
lora_v_proj = 3072 * r + 1024 * r
lora_o_proj = 3072 * r + 3072 * r
lora_gate_proj = 3072 * r + 3072 * r
lora_up_proj   =3072 * r + 1024 * r
lora_down_proj = 3072 * r + 3072 * r

lora_layer = (
    lora_q_proj
    + lora_k_proj
    + lora_v_proj
    + lora_o_proj
   + lora_gate_proj +
lora_up_proj +
lora_down_proj
)

params = lora_layer * 28

size = (params * 4) / 1_000_000

print(
    f"Total number of params: {params:,} "
    f"and size {size:,.1f}MB"
)

Total number of params: 264,241,152 and size 1,057.0MB
